In [1]:
import warnings
# Ignore all warnings

# Enable interactive mode with a scrollable plot
%matplotlib qt
import numpy as np
import glob
import matplotlib.pyplot as plt
import sys
import pickle as pkl
sys.path.append("..")

import mne
from mne.io import RawArray
from mne.channels import make_standard_montage
from mne.preprocessing import read_ica
from mne.io import read_raw_fif
import gc
import fooof
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel

# import mne_microstates
import pandas as pd
import seaborn as sns
import matplotlib.patches as mpatches

import os
from scipy.io import loadmat
import scipy

import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap
import pyxdf
import socket
import pickle as pkl
from pyprep.find_noisy_channels import NoisyChannels
import copy
from pathlib import Path
import nibabel as nib
import json
import nibabel as nib
from nilearn.image import new_img_like
from nilearn.plotting import plot_img_on_surf
from matplotlib.colors import ListedColormap

%load_ext autoreload
%autoreload 2

C:\Users\specsdcn\AppData\Local\Temp\ipykernel_20272\2543603797.py:19: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [2]:
device_name = socket.gethostname()
print(device_name)

DESKTOP-I0FNO1F


In [3]:
if device_name == "LAPTOP-LRI6535I":
    path_analysis_data = "E:/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/analysis_plots/"

if device_name == "kritrim":
    path_analysis_data = "C:/Users/manda/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"

if device_name == "DESKTOP-I0FNO1F":
    path_analysis_data = "D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"



In [4]:
from helpers.source_level_psd_helpers import *
from helpers.frequency_analysis import compute_psd_mne
from helpers.helper_functions import get_matched_legths
from helpers.topo_eeg import get_fdr_p_values, get_band_power_difference_all_subjects_significance_aperiodic_fdr, get_channel_wise_fooof
from helpers.helper_functions import get_fooof_aperiodic, get_fooof
from helpers.group_level_helpers import group_level_bootstrap_test_network, build_channel_style_arrays, latex_group_vs_individual_table

def store_mne_raw(eeg_filtered, path_folder, path_data, ovwrite=False):
    eeg_filtered.save(path_folder + path_data, overwrite=ovwrite)

def load_mne_raw(path_folder, path_data):
    return mne.io.read_raw_fif(path_folder + path_data, preload=True)


Python Executable Path: 
c:\Users\specsdcn\anaconda3\envs\pdoc\python.exe


In [5]:
import warnings
warnings.filterwarnings('ignore')

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}



pat_state_dict = {
    'sub01': 'UWS',
    'sub03': 'MCS+',
    'sub04': 'MCS-',
    'sub07': 'MCS-',
    'sub09': 'MCS-',
    'sub10': 'MCS-',
    'sub12': 'MCS-',
}



sfreq = 512
paradigms = {0: "noise", 1: "music"}
pat_state = {"sub01": "UWS", "sub03": "MCS+", "sub04": "MCS-", "sub07":"MCS-", "sub09":"MCS-", "sub10":"MCS-", "sub12":"MCS-"}


#### Load AAL Atlas

In [6]:
aal_dir = path_analysis_data_new + "eeg_source/"


with open(f'{aal_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [7]:
aal_labels

['Precentral_L',
 'Precentral_R',
 'Frontal_Sup_L',
 'Frontal_Sup_R',
 'Frontal_Sup_Orb_L',
 'Frontal_Sup_Orb_R',
 'Frontal_Mid_L',
 'Frontal_Mid_R',
 'Frontal_Mid_Orb_L',
 'Frontal_Mid_Orb_R',
 'Frontal_Inf_Oper_L',
 'Frontal_Inf_Oper_R',
 'Frontal_Inf_Tri_L',
 'Frontal_Inf_Tri_R',
 'Frontal_Inf_Orb_L',
 'Frontal_Inf_Orb_R',
 'Rolandic_Oper_L',
 'Rolandic_Oper_R',
 'Supp_Motor_Area_L',
 'Supp_Motor_Area_R',
 'Olfactory_L',
 'Olfactory_R',
 'Frontal_Sup_Medial_L',
 'Frontal_Sup_Medial_R',
 'Frontal_Med_Orb_L',
 'Frontal_Med_Orb_R',
 'Rectus_L',
 'Rectus_R',
 'Insula_L',
 'Insula_R',
 'Cingulum_Ant_L',
 'Cingulum_Ant_R',
 'Cingulum_Mid_L',
 'Cingulum_Mid_R',
 'Cingulum_Post_L',
 'Cingulum_Post_R',
 'Hippocampus_L',
 'Hippocampus_R',
 'ParaHippocampal_L',
 'ParaHippocampal_R',
 'Amygdala_L',
 'Amygdala_R',
 'Calcarine_L',
 'Calcarine_R',
 'Cuneus_L',
 'Cuneus_R',
 'Lingual_L',
 'Lingual_R',
 'Occipital_Sup_L',
 'Occipital_Sup_R',
 'Occipital_Mid_L',
 'Occipital_Mid_R',
 'Occipital_Inf_L'

#### Load Source Reconstructed Data

In [8]:
with open(path_analysis_data_new + "all_eeg_source_reconstruction_eloreta_45_adhoccov.pkl", "rb") as f:
    eeg_source_reconstructed = pkl.load(f)


In [9]:
eeg_source_reconstructed["sub01"]["rest_init"].shape

(80, 116, 2560)

In [10]:
eeg_source_reconstructed_cont = {}

for sub in eeg_source_reconstructed:
    eeg_source_reconstructed_cont[sub] = {}
    for parad in  eeg_source_reconstructed[sub]:
        eeg_source_reconstructed_cont[sub][parad] = eeg_source_reconstructed[sub][parad].transpose(1, 0, 2).reshape(116, -1)


In [11]:
a = np.empty_like(eeg_source_reconstructed_cont[sub][parad])
a.shape

(116, 171520)

In [12]:
count_cerebrum_labels = 0 

for label in aal_labels:

    if label.startswith("Cerebelum") or label.startswith("Vermis"):
        continue

    else:
        count_cerebrum_labels += 1

count_cerebrum_labels

90

In [13]:
eeg_source_reconstructed_cont_cerebrum = {}
eeg_source_reconstructed_cont_cerebrum_raw = {}

regions_all = eeg_source_reconstructed_cont[sub][parad].shape[0]

n_regions = 90
ch_types = ['eeg'] * n_regions

info = mne.create_info(ch_names=ch_types, sfreq=512, ch_types=ch_types)

for sub in eeg_source_reconstructed_cont:
    eeg_source_reconstructed_cont_cerebrum[sub] = {}
    eeg_source_reconstructed_cont_cerebrum_raw[sub] = {}

    for parad in  eeg_source_reconstructed_cont[sub]:
        eeg_source_reconstructed_cont_cerebrum[sub][parad] = np.zeros([count_cerebrum_labels, eeg_source_reconstructed_cont[sub][parad].shape[1]])

        for reg in range(regions_all):
            reg_label = aal_labels[reg]

            if reg_label.startswith("Cerebelum") or reg_label.startswith("Vermis"):
                continue

            else:
                eeg_source_reconstructed_cont_cerebrum[sub][parad][reg, :] = eeg_source_reconstructed_cont[sub][parad][reg, :]

        eeg_source_reconstructed_cont_cerebrum_raw[sub][parad] = mne.io.RawArray(eeg_source_reconstructed_cont_cerebrum[sub][parad], info, verbose=False)

In [14]:
eeg_source_reconstructed_cont_cerebrum_raw['sub07'].keys()

dict_keys(['rest_init', 'reward', 'pink_segs', 'music_segs', 'rest_mid', 'rest_end'])

FOOOF

In [15]:
band_powers_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}
psds_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}

subjects = list(eeg_source_reconstructed_cont_cerebrum.keys())
paradigms = list(eeg_source_reconstructed_cont_cerebrum[subjects[0]].keys())


f_min = bands["delta"][0]
f_max = bands["gamma low"][1]

segment_parads = ['pink_segs', 'music_segs']
power_type = "relative"

for sub in eeg_source_reconstructed_cont_cerebrum:
    for parad in  eeg_source_reconstructed_cont_cerebrum[sub]:

        # if parad in segment_parads:
        #     segment_secs = 3

        # else:
        segment_secs = 5
        
        segment_size = int(sfreq * segment_secs)
        overlap = int(segment_size /2)

        eeg_array = eeg_source_reconstructed_cont_cerebrum[sub][parad]
        n_regions = eeg_array.shape[0]
        ch_types = ['eeg'] * n_regions

        info = mne.create_info(ch_names=ch_types, sfreq=sfreq, ch_types=ch_types)
        eeg_raw = mne.io.RawArray(eeg_array, info, verbose=False)

        band_powers, psds, freqs = compute_psd_mne(eeg_raw, bands, overlap, segment_size, f_min, f_max, power_type=power_type)

        band_powers_all[sub][parad] = band_powers
        psds_all[sub][parad] = [psds, freqs]

Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective wind

In [16]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_new.pkl", "wb") as f:
#     pkl.dump(psds_all, f)

In [17]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_new.pkl", "rb") as f:
#     psds_all = pkl.load(f)

In [18]:
psds_all.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub10', 'sub12', 'sub09'])

In [19]:
# eeg_fooof_all = get_fooof(psds_all, save_path=path_analysis_data_new + "fooof_all_sources_adhoccov/")

LOAD FOOOF

In [20]:
fooof_path = path_analysis_data_new + "fooof_all_sources_adhoccov/"
fooof_files = glob.glob(fooof_path + "*.pkl")

In [21]:
from pathlib import Path
eeg_fooof_all_subs = {sub:{} for sub in eeg_source_reconstructed_cont_cerebrum}

for fl in fooof_files:
    file_stem = Path(fl).stem
    file_name = Path(fl).name

    sub_name = file_stem.split("_")[2][:5]
    parad_name = file_stem.split("_")[2][5:]

    try:
        parad_name_2 = file_stem.split("_")[3]
        parad_name += "_" + parad_name_2

    except:
        parad_name = parad_name
    
    print(sub_name, parad_name)
    
    with open(fl, "rb") as f:
        eeg_fooof_all_subs[sub_name][parad_name] = pkl.load(f)
        f.close()
        

sub01 rest_init
sub01 reward
sub03 rest_init
sub03 reward
sub04 rest_init
sub04 reward
sub07 music_segs
sub07 pink_segs
sub07 rest_end
sub07 rest_init
sub07 rest_mid
sub07 reward
sub09 music_segs
sub09 pink_noise
sub09 pink_segs
sub09 rest_end
sub09 rest_init
sub09 rest_mid
sub09 reward
sub10 music_segs
sub10 pink_noise
sub10 pink_segs
sub10 rest_init
sub10 rest_mid
sub10 reward
sub12 music_segs
sub12 pink_noise
sub12 pink_segs
sub12 rest_end
sub12 rest_init
sub12 rest_mid
sub12 reward


### Plot the Differences in FOOOF between Conditions

Import AAL Atlas

In [22]:
data_dir = path_analysis_data_new + "eeg_source/"

with open(f'{data_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [23]:
data_dir = path_analysis_data_new + "eeg_source/"
aal_nii_path = f'{data_dir}AAL.nii'   # adjust filename if needed
aal_img = nib.load(aal_nii_path)

aal_data = np.squeeze(np.asanyarray(aal_img.dataobj)).astype(int)
print("AAL Atlas Shape: ", aal_data.shape)

AAL Atlas Shape:  (91, 109, 91)


In [24]:
aal_regions = np.unique(aal_data)
aal_regions = aal_regions[aal_regions != 0]
aal_regions

data_dict = {
    "id": [],
    "regions": [],
    "labels": []
}

with open(path_analysis_data_new + "eeg_source/aal1_for_SPM8/ROI_MNI_V4.txt", "r") as f:
    for line in f:
        if line.strip():
            parts = line.split()  # split by whitespace
            
            data_dict["id"].append(parts[0])
            data_dict["regions"].append(parts[1])
            data_dict["labels"].append(parts[2])

print(data_dict.keys())


aal_regions_str = [str(reg) for reg in aal_regions]

count_reg_similar = 0

for reg in aal_regions_str:
    if reg in data_dict["labels"]:
        count_reg_similar += 1

print("Similar Regions Number: ", count_reg_similar)


label_region_mapping = {}

for indx in range(len(data_dict["labels"])):
    label_region_mapping[int(data_dict["labels"][indx])] = data_dict["regions"][indx]


dict_keys(['id', 'regions', 'labels'])
Similar Regions Number:  116


#### 1. Rest vs. Reward

In [25]:
eeg_fooof_rest_reward = {}


eeg_fooof_rest_reward["sub01"] = {}
eeg_fooof_rest_reward["sub01"]["rest_init"] = eeg_fooof_all_subs["sub01"]["rest_init"].copy()
eeg_fooof_rest_reward["sub01"]["reward"] = eeg_fooof_all_subs["sub01"]["reward"].copy()

eeg_fooof_rest_reward["sub03"] = {}
eeg_fooof_rest_reward["sub03"]["rest_init"] = eeg_fooof_all_subs["sub03"]["rest_init"].copy()
eeg_fooof_rest_reward["sub03"]["reward"] = eeg_fooof_all_subs["sub03"]["reward"].copy()

eeg_fooof_rest_reward["sub04"] = {}
eeg_fooof_rest_reward["sub04"]["rest_init"] = eeg_fooof_all_subs["sub04"]["rest_init"].copy()
eeg_fooof_rest_reward["sub04"]["reward"] = eeg_fooof_all_subs["sub04"]["reward"].copy()

eeg_fooof_rest_reward["sub07"] = {}
eeg_fooof_rest_reward["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_reward["sub09"] = {} 
eeg_fooof_rest_reward["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_reward["sub10"] = {} 
eeg_fooof_rest_reward["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_reward["sub12"] = {} 
eeg_fooof_rest_reward["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [26]:
eeg_fooof_rest_reward_matched = get_matched_legths(eeg_fooof_rest_reward)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 159 epochs
(90, 223, 159) (90, 223, 159)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 153 epochs
(90, 223, 153) (90, 223, 153)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 69 epochs
(90, 223, 69) (90, 223, 69)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 145 epochs
(90, 223, 145) (90, 223, 145)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 175 → 133 epochs (dropped last 42)
(90, 223, 133) (90, 223, 133)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 95 → 85 epochs (dropped last 10)
(90, 223, 85) (90, 223, 85)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 139 → 105 epochs (dropped last 34)
(90, 223, 105) (90, 223, 105)


In [27]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

_, rest_init_music_fooof_bands_pvalues_abs, fooof_rest_init_music = get_channel_wise_fooof(freqs, eeg_fooof_rest_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals = get_fdr_p_values(rest_init_music_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)
     #0 for absolute and 1 for relative power
    

sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward


In [28]:
raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_rest_init_music, cond1_name="rest_init",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=0) #0 for absolute and 1 for relative power

✅ Done! Used Functional Network clustering
Number of clusters: 8


In [29]:
def filter_networks(cluster_means, networks_to_keep):
    """
    Return a new dict containing only the specified networks, preserving the
    subject -> condition -> band -> network structure.
    """
    import copy

    out = copy.deepcopy(cluster_means)

    for sub in out:
        for cond in out[sub]:
            for band in out[sub][cond]:
                out[sub][cond][band] = {
                    net: out[sub][cond][band][net]
                    for net in networks_to_keep
                    if net in out[sub][cond][band]
                }

    return out


In [30]:
cluster_means["sub01"]["rest_init"]["delta"].keys()

dict_keys(['Sensorimotor Network', 'Dorsal Attention Network', 'Limbic Network', 'Frontoparietal Network', 'Default Mode Network', 'Ventral Attention Network', 'Visual Network', 'Basal Ganglia Network'])

In [31]:
cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]
)

In [35]:
plots_path_power = plots_path + "fdr_source/fc_powers_rest_music_cont_all/"

pval_rest_reward = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Rest", legend_label2="Music Cont.", save_dir=plots_path_power, save=True, subject_order=None, 
    pat_state=pat_state_dict, vertical=True,  FS_PVAL = 20, FS_YLAB = 26, FS_TITLE = 26, show_ylabel=False)

stack_subject_figures_pdf(folder_path=plots_path_power, stack_mode="vertical", output_path=plots_path_power + "merged_all/src_violin_rest_musiccont.pdf", fig_width=18)

Running bootstrap tests …


KeyError: 'delta'

In [33]:
mcs_minus_subjects = ["sub04", "sub07", "sub09", "sub10", "sub12"]
networks = [
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="rest_init",
    condition2="reward",
    bands=bands,
    networks=networks,
)

networks = ["Sensorimotor Network", "Default Mode Network", "Limbic Network"]

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pval_rest_reward,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)


latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Rest vs. Continuous Music",
    mode='start',
    n_channels=len(networks),  # 3, not 128
    caption="Group-level versus individual-level significance across functional networks and frequency bands.",
    label="tab:group_vs_individual_network",
)

print(latex_code)

\begin{table}[htbp]
\centering
\caption{Group-level versus individual-level significance across functional networks and frequency bands.}
\label{tab:group_vs_individual_network}

\begin{tabularx}{\textwidth}{@{}l l X X X X@{}}
\toprule
\textbf{Comparison} &
\textbf{Band} &
\textbf{Group-level sig.} &
\textbf{Sig. in $\geq$1 subject} &
\textbf{Sig. in majority} &
\textbf{Sig. in all subjects} \\
\midrule

\multirow{5}{*}{\shortstack[l]{Rest vs.\\Continuous Music}}
& $\delta$ & 0/3 & 3/3 & 0/3 ($\geq$3/5) & 0/3 \\
& $\theta$ & 0/3 & 3/3 & 2/3 ($\geq$3/5) & 0/3 \\
& $\alpha$ & 0/3 & 3/3 & 3/3 ($\geq$3/5) & 1/3 \\
& $\beta$ & 0/3 & 3/3 & 3/3 ($\geq$3/5) & 0/3 \\
& $\gamma$ & 0/3 & 3/3 & 3/3 ($\geq$3/5) & 1/3 \\

\bottomrule
\end{tabularx}
\end{table}



In [34]:
# plots_path_power = plots_path + "fdr_source/powers_rest_reward_all/anatomical/"

# raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_rest_init_music, cond1_name="rest_init",cond2_name="reward",
#     mapping_dict=region_to_anatomical_cluster, is_anatomical=True, avg_method='mean', alpha=0.05, abs_rel=1)

# plot_band_power_comparisons(cluster_means=cluster_means, condition1="rest_init", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
#     alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), save_dir=plots_path_power,
# )

# stack_subject_figures(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/merged.png",
#     fig_width=28, row_height=28)

Aperiodic

In [33]:
eeg_fooof_rest_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_reward_matched)
fdr_corrected_p_vals_rest_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_reward_aperiodic, pat_state, n_channels=n_regions)



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


In [36]:

raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_rest_reward_aperiodic, cond1_name="rest_init",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2) #2 means aperiodic power, 0 for absolute and 1 for relative power

    
cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])
        
            
            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # # elif sub == "sub02":
            # #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] - 10)
            # #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # else:
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]
)


plots_path_power = plots_path + "fdr_source/fc_powers_rest_music_cont_all/ap/"

save_name = ""

plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(3, 5), save=True, legend_label1="Rest", legend_label2="Music Cont.", save_dir=plots_path_power, 
    power_unit=r"Aperiodic Parameters (a.u.)", save_tag=save_name, subject_order=None, pat_state=pat_state_dict, show_legend=False, vertical=True,
    FS_PVAL = 20, FS_YLAB = 26, FS_TITLE = 26, show_ylabel=False) #FS_PVAL = 18, FS_YLAB = 12, FS_BAND = 22, FS_TITLE = 22)




stack_subject_figures_pdf(folder_path=plots_path_power, stack_mode="vertical", 
                          output_path=plots_path_power + "merged_all/src_aperiodic_violin_rest_musiccont.pdf", fig_width=9)

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_rest_music_cont_all/ap/merged_all/legend_rest_init_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_rest_music_cont_all/ap/merged_all/src_aperiodic_violin_rest_musiccont.pdf


#### 2. Music Segs vs. Pink Segs

In [37]:
eeg_fooof_music_pink_segs = {}


eeg_fooof_music_pink_segs["sub07"] = {}
eeg_fooof_music_pink_segs["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub07"]["music_segs"] = eeg_fooof_all_subs["sub07"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub09"] = {} 
eeg_fooof_music_pink_segs["sub09"]["pink_segs"] = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub09"]["music_segs"] = eeg_fooof_all_subs["sub09"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub10"] = {} 
eeg_fooof_music_pink_segs["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub10"]["music_segs"] = eeg_fooof_all_subs["sub10"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub12"] = {} 
eeg_fooof_music_pink_segs["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub12"]["music_segs"] = eeg_fooof_all_subs["sub12"]["music_segs"].copy()


eeg_fooof_music_pink_segs_matched = get_matched_legths(eeg_fooof_music_pink_segs)

Subject: sub07, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 255 → 249 epochs (dropped last 6)
(90, 223, 249) (90, 223, 249)
Subject: sub09, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 187 → 185 epochs (dropped last 2)
(90, 223, 185) (90, 223, 185)
Subject: sub10, Paradigms: ['pink_segs', 'music_segs']
Truncated array2 from 135 → 69 epochs (dropped last 66)
(90, 223, 69) (90, 223, 69)
Subject: sub12, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 165 → 163 epochs (dropped last 2)
(90, 223, 163) (90, 223, 163)


In [38]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

_, music_pink_seg_fooof_bands_pvalues_abs, fooof_music_pink_segs = get_channel_wise_fooof(freqs, eeg_fooof_music_pink_segs_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_music_pink_segs = get_fdr_p_values(music_pink_seg_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub07
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub09
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub10
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub12
Paradigm 0 pink_segs
Paradigm 1 music_segs


In [39]:
plots_path_power = plots_path + "fdr_source/fc_powers_music_segs_pink_segs_all/"

raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_music_pink_segs, cond1_name="pink_segs",cond2_name="music_segs",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=0)



cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]
)


pvals_music_pink_segs = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink N. Segs", legend_label2="Music Segs", save=True, save_dir=plots_path_power, subject_order=None, 
    pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20
)

stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_violin_musicsegs_pinksegs.pdf")

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_segs_pink_segs_all/merged_all/legend_pink_segs_vs_music_segs_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_segs_pink_segs_all/merged_all/src_violin_musicsegs_pinksegs.pdf


In [40]:
mcs_minus_subjects = ["sub07", "sub09", "sub10", "sub12"]
networks = [
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="pink_segs",
    condition2="music_segs",
    bands=bands,
    networks=networks,
)

networks = [
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pvals_music_pink_segs,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)

latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Music Segs. vs. Pink Noise Segs.",
    mode='append',
    n_channels=len(networks),  # 3, not 128
    
)

print(latex_code)

\midrule

\multirow{5}{*}{\shortstack[l]{Music Segs. vs.\\Pink Noise Segs.}}
& $\delta$ & 0/3 & 0/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\theta$ & 0/3 & 1/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\alpha$ & 0/3 & 0/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\beta$ & 0/3 & 0/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\gamma$ & 0/3 & 2/3 & 0/3 ($\geq$2/4) & 0/3 \\




Aperiodic

In [49]:
eeg_fooof_music_pink_segs_aperiodic = get_fooof_aperiodic(eeg_fooof_music_pink_segs_matched)

fdr_corrected_p_vals_music_pink_segs_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_music_pink_segs_aperiodic, pat_state, n_channels=n_regions)


raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_music_pink_segs_aperiodic, cond1_name="pink_segs",cond2_name="music_segs",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2)


cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])
            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
        # 'Dorsal Attention Network',
        # 'Frontoparietal Network',
        # 'Ventral Attention Network'
    ]
)


plots_path_power = plots_path + "fdr_source/fc_powers_music_segs_pink_segs_all/ap/"

save_name = ""

p_Val_music_pink_seg = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink N. Segs", legend_label2="Music Segs", power_unit=r"Aperiodic Parameters (a.u.)", 
    show_legend=False, save_dir=plots_path_power, save=True, save_tag=save_name, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20)


stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_aperiodic_violin_musicsegs_pinksegs.pdf")

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_segs_pink_segs_all/ap/merged_all/legend_pink_segs_vs_music_segs_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_segs_pink_segs_all/ap/merged_all/src_aperiodic_violin_musicsegs_pinksegs.pdf


In [50]:
p_Val_music_pink_seg["sub10"]["Offset"]["Limbic Network"]

np.float64(0.04995004995004995)

#### 3. Reward vs. Pink Noise

In [42]:
eeg_fooof_pink_noise_reward = {}

eeg_fooof_pink_noise_reward["sub09"] = {} 
eeg_fooof_pink_noise_reward["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub10"] = {} 
eeg_fooof_pink_noise_reward["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub12"] = {}
eeg_fooof_pink_noise_reward["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

eeg_fooof_pink_noise_reward_matched = get_matched_legths(eeg_fooof_pink_noise_reward)

Subject: sub09, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 133 → 101 epochs (dropped last 32)
(90, 223, 101) (90, 223, 101)
Subject: sub10, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 85 → 71 epochs (dropped last 14)
(90, 223, 71) (90, 223, 71)
Subject: sub12, Paradigms: ['pink_noise', 'reward']
Truncated array1 from 115 → 105 epochs (dropped last 10)
(90, 223, 105) (90, 223, 105)


In [43]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90
    
_, pink_noise_reward_fooof_bands_pvalues_abs, fooof_music_pink_noise= get_channel_wise_fooof(freqs, eeg_fooof_pink_noise_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_pink_noise_reward = get_fdr_p_values(pink_noise_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub09
Paradigm 0 pink_noise
Paradigm 1 reward
sub10
Paradigm 0 pink_noise
Paradigm 1 reward
sub12
Paradigm 0 pink_noise
Paradigm 1 reward


In [44]:
plots_path_power = plots_path + "fdr_source/fc_powers_music_cont_pink_noise_cont_all/"

raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_music_pink_noise, cond1_name="pink_noise",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=1)


cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
    ],
)

pvals_pink_reward = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink Noise Cont.", legend_label2="Music Cont.", save=True, save_dir=plots_path_power, 
    subject_order=None, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20
)

stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_violin_pinkcont_musiccont.pdf" )

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_cont_pink_noise_cont_all/merged_all/legend_pink_noise_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_cont_pink_noise_cont_all/merged_all/src_violin_pinkcont_musiccont.pdf


In [45]:
mcs_minus_subjects = ["sub09", "sub10", "sub12"]
networks = ["Sensorimotor Network", "Default Mode Network", "Limbic Network"]

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="pink_noise",
    condition2="reward",
    bands=bands,
    networks=networks,
)

networks = ["Sensorimotor Network", "Default Mode Network", "Limbic Network"]

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pvals_pink_reward,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)

latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Pink Noise Cont. vs. Music Cont.",
    mode='append',
    n_channels=len(networks),  # 3, not 128
    
)

print(latex_code)

\midrule

\multirow{5}{*}{\shortstack[l]{Pink Noise Cont. vs.\\Music Cont.}}
& $\delta$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\theta$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\alpha$ & 0/3 & 2/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\beta$ & 0/3 & 2/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\gamma$ & 0/3 & 3/3 & 0/3 ($\geq$2/3) & 0/3 \\




Aperiodic

In [46]:
eeg_fooof_pink_noise_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_pink_noise_reward_matched)

fdr_corrected_p_vals_pink_noise_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_pink_noise_reward_aperiodic, pat_state, n_channels=n_regions)



raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_pink_noise_reward_aperiodic , cond1_name="pink_noise",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2)




cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=[
        "Sensorimotor Network",
        "Default Mode Network",
        "Limbic Network",
    ],
)


plots_path_power = plots_path + "fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/"

save_name = ""

plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink Noise Cont.", legend_label2="Music Cont.", save_dir=plots_path_power, 
    show_legend=False, power_unit=r"Aperiodic Parameters (a.u.)", save=True, save_tag=save_name, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20)


stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_aperiodic_violin_pinkcont_musiccont.pdf")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/merged_all/legend_pink_noise_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/merged_all/src_aperiodic_violin_pinkcont_musiccont.pdf
